# 演習5. スレッドセーフなキューを組み立てる

## シーン

演習4で、待ち合わせの形は分かりました。

```cpp
// 入れる側
{ std::lock_guard<std::mutex> g(mtx); q.push(v); }
can_pop.notify_one();

// 取り出す側
std::unique_lock<std::mutex> lk(mtx);
can_pop.wait(lk, [] { return !q.empty(); });
v = q.front(); q.pop();
```

正しく書けば動きます。**しかし、これを使うたびに毎回書くのは無理があります。**

- `notify` を1か所でも書き忘れれば固まります（演習4の発展課題5）
- 述語を1か所でも書き忘れれば、空のキューを触ります（4-4）
- 鍵の外で `q` を触るコードが1行紛れ込めば、競合します（演習2）

**使う側が間違えられないようにする**のが、この演習の目的です。
鍵も条件変数もクラスの中に閉じ込めて、外からは `push` と `pop` しか見えないようにします。

3段階で組み立てます。

- **第1版** … 鍵をかけただけ
- **第2版** … 待てるようにする
- **完成版** … 容量の上限を付ける

出来上がるものは、本番のプログラムで使われているキューとほぼ同じ形です。

## 5-1. 第1版 ―― 鍵をかけただけのキュー

まず、演習3までの知識だけで書ける版です。

```cpp
template <typename T>
class SimpleQueue {
public:
    void push(const T& v) { std::lock_guard<std::mutex> g(mtx_); q_.push(v); }

    bool pop(T& out) {                       // 取り出せたら true、空なら false
        std::lock_guard<std::mutex> g(mtx_);
        if (q_.empty()) return false;
        out = q_.front(); q_.pop();
        return true;
    }

private:
    std::queue<T> q_;                        // 中身
    std::mutex mtx_;                         // 鍵
};
```

`push` も `pop` も鍵の中なので、**競合はしません。壊れません。**
`q_` と `mtx_` を `private:` に置いてあるので、外から直接触ることもできません。

**では、これで十分でしょうか。** 実行して確かめます。

In [ ]:
%%writefile ex05a.cpp
#include <iostream>
#include <thread>
#include <queue>
#include <mutex>
#include <chrono>
using namespace std::chrono;

// ---- 第1版：鍵をかけただけのキュー ----
template <typename T>
class SimpleQueue {
public:
    void push(const T& v) {
        std::lock_guard<std::mutex> g(mtx_);
        q_.push(v);
    }
    // 取り出せたら true、空なら false を返す（待たない）
    bool pop(T& out) {
        std::lock_guard<std::mutex> g(mtx_);
        if (q_.empty()) return false;
        out = q_.front(); q_.pop();
        return true;
    }
private:
    std::queue<T> q_;
    std::mutex mtx_;
};

SimpleQueue<int> q;
long failed = 0;                      // pop が空振りした回数

void producer() {
    for (int i = 1; i <= 5; i++) {
        std::this_thread::sleep_for(milliseconds(200));
        q.push(i);
    }
}

void consumer() {
    for (int n = 0; n < 5; n++) {
        int v;
        while (!q.pop(v)) {           // ← 空振りしたら、また呼び直すしかない
            failed++;
            std::this_thread::sleep_for(milliseconds(1));
        }
        std::cout << "受け取った : " << v << "\n";
    }
}

int main() {
    std::thread p(producer), c(consumer);
    p.join(); c.join();
    std::cout << "\npop が空振りした回数 = " << failed << " 回\n";
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ex05a.cpp -o ex05a && ./ex05a

### 5-1-1. 結果 ―― 壊れないが、待てない

受け取る側のコードを見てください。

```cpp
while (!q.pop(v)) {                                  // 空振りしたら
    failed++;
    std::this_thread::sleep_for(milliseconds(1));    // 少し眠って、また呼ぶ
}
```

**演習4-2 のポーリングが戻ってきています。** 空振りが数百回。

第1版の問題は「壊れること」ではありません。
**「空だった」という後始末を、使う側に押し付けていること**です。
押し付けられた側は、結局ポーリングを書くしかありません。

> **キューが「待つ」を引き受けなければ、使う側が必ずポーリングを書くことになる。**

`bool` を返す形の `pop` そのものが悪いわけではありません
（「待ちたくない、空なら空でよい」という使い方もあります）。
足りないのは「**空なら待つ**」という選択肢のほうです。

## 5-2. 第2版 ―― 待てるようにする

演習4で覚えた形を、そのままクラスの中に入れます。

```cpp
void push(const T& v) {
    { std::lock_guard<std::mutex> g(mtx_); q_.push(v); }
    can_pop_.notify_one();                                     // 入れたら必ず起こす
}

T pop() {
    std::unique_lock<std::mutex> lk(mtx_);
    can_pop_.wait(lk, [this] { return !q_.empty(); });         // 空なら待つ
    T v = q_.front(); q_.pop();
    return v;
}
```

`pop()` は**失敗しません**。空なら、入るまで待つからです。
使う側は `int v = q.pop();` と書くだけでよくなりました。

> **`[this]` はラムダ式の捕獲です**（C++問題集の**問7**）。
> `q_` はクラスのメンバなので、ラムダの中から使うには
> 「自分自身（`this`）を捕まえる」と書く必要があります。
> 本番のプログラムでも、この `[this]` がそのまま出てきます。

観察のため、**キューに並んだ最大の個数**も数えておきます。
作る側を受け取る側より5倍速くして走らせます。

In [ ]:
%%writefile ex05b.cpp
#include <iostream>
#include <thread>
#include <queue>
#include <mutex>
#include <condition_variable>
#include <chrono>
using namespace std::chrono;

// ---- 第2版：待てるキュー（容量はまだ無い） ----
template <typename T>
class WaitingQueue {
public:
    void push(const T& v) {
        {
            std::lock_guard<std::mutex> g(mtx_);
            q_.push(v);
            if (q_.size() > peak_) peak_ = q_.size();     // 観察用：最大の行列の長さ
        }
        can_pop_.notify_one();                            // 鍵を開けてから起こす
    }
    T pop() {                                             // 空なら「待つ」。失敗しない
        std::unique_lock<std::mutex> lk(mtx_);
        can_pop_.wait(lk, [this] { return !q_.empty(); });
        T v = q_.front(); q_.pop();
        return v;
    }
    std::size_t peak() const { return peak_; }
private:
    std::queue<T> q_;
    std::size_t peak_ = 0;
    std::mutex mtx_;
    std::condition_variable can_pop_;
};

WaitingQueue<int> q;

void producer(int n, int make_ms) {
    for (int i = 1; i <= n; i++) {
        std::this_thread::sleep_for(milliseconds(make_ms));
        q.push(i);
    }
}

void consumer(int n, int use_ms) {
    for (int i = 0; i < n; i++) {
        int v = q.pop();                                  // ← ただ呼ぶだけ
        std::this_thread::sleep_for(milliseconds(use_ms));
        (void)v;
    }
}

int main() {
    // 作る側は 10ms に1個、受け取る側は 1個 50ms かかる（作る側のほうが5倍速い）
    auto t0 = steady_clock::now();
    std::thread p(producer, 30, 10), c(consumer, 30, 50);
    p.join(); c.join();
    std::cout << "30個を処理するのにかかった時間 = "
              << duration_cast<milliseconds>(steady_clock::now() - t0).count() << " ms\n";
    std::cout << "キューに並んだ最大の個数 = " << q.peak() << " 個\n";
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ex05b.cpp -o ex05b && ./ex05b

### 5-2-1. 結果 ―― ポーリングは消えた。しかし行列が伸びる

空振りはゼロになりました。使う側のコードもきれいです。

ところが「並んだ最大の個数」を見てください。**30個中 24個** が
キューに溜まっていました。作る側のほうが速いので、当然です。

```
作る係     10ms に1個 ─▶  [ キュー ]  ─▶  受け取る係  1個 50ms
                            ↑
                     どんどん伸びていく
```

いまは 30個で終わるので 24個で済んでいますが、**これが動画だったらどうなるでしょうか。**

- 1フレームが 512×256×3バイト ≒ **384KB**
- 毎秒30フレーム、受け取る側が半分の速さしか出ないなら、毎秒15フレームずつ溜まる
- 1分で 900フレーム ≒ **340MB**。10分で 3.4GB

**メモリを食い尽くして落ちます。** そして落ちる前から、別の問題も起きています。
キューに1000フレーム並んでいたら、いま入れたフレームが画面に出るのは**33秒後**です。
リアルタイム処理としては、すでに壊れています。

> **キューが伸びるということは、メモリと「遅れ」が伸びるということ。**

## 5-3. 【予測クイズ】完成版 ―― 容量の上限を付ける

対策は単純です。**キューに上限を決めて、満杯なら入れる側を待たせます。**

`pop` が「空なら待つ」のと、まったく同じ形を `push` にも付けるだけです。
そのために条件変数がもう1本要ります。

- `can_pop_` … 「取り出せるようになった」（空でなくなった）
- `can_push_` … 「入れられるようになった」（満杯でなくなった）

これで完成版です。さきほどとまったく同じ速さの作る係・受け取る係で、
**容量を3**にして走らせます。

**実行する前に予測してください。**

- 30個を処理し終える時間は、第2版（約1500ms）と比べてどうなるでしょうか。
  速くなる？ 遅くなる？ 変わらない？
- キューに並んだ最大の個数はいくつになるでしょうか

In [ ]:
%%writefile ex05c.cpp
#include <iostream>
#include <thread>
#include <queue>
#include <mutex>
#include <condition_variable>
#include <chrono>
using namespace std::chrono;

// ---- 完成版：スレッドセーフで、容量の上限があるキュー ----
template <typename T>
class BoundedQueue {
public:
    explicit BoundedQueue(std::size_t capacity) : capacity_(capacity) {}

    void push(const T& v) {
        std::unique_lock<std::mutex> lk(mtx_);
        can_push_.wait(lk, [this] { return q_.size() < capacity_; });   // 満杯なら待つ
        q_.push(v);
        if (q_.size() > peak_) peak_ = q_.size();
        lk.unlock();                    // 先に鍵を開けてから
        can_pop_.notify_one();          // 取り出したい人を起こす
    }

    T pop() {
        std::unique_lock<std::mutex> lk(mtx_);
        can_pop_.wait(lk, [this] { return !q_.empty(); });              // 空なら待つ
        T v = q_.front(); q_.pop();
        lk.unlock();                    // 先に鍵を開けてから
        can_push_.notify_one();         // 入れたい人を起こす
        return v;
    }

    std::size_t size() const { std::lock_guard<std::mutex> g(mtx_); return q_.size(); }
    std::size_t peak() const { return peak_; }

private:
    std::queue<T> q_;
    std::size_t capacity_;
    std::size_t peak_ = 0;
    mutable std::mutex mtx_;            // size() は const なので mutable が要る
    std::condition_variable can_pop_;   // 「取り出せるようになった」
    std::condition_variable can_push_;  // 「入れられるようになった」
};

BoundedQueue<int> q(3);                 // ← 容量3

void producer(int n, int make_ms) {
    for (int i = 1; i <= n; i++) {
        std::this_thread::sleep_for(milliseconds(make_ms));
        q.push(i);
    }
}

void consumer(int n, int use_ms) {
    for (int i = 0; i < n; i++) {
        int v = q.pop();
        std::this_thread::sleep_for(milliseconds(use_ms));
        (void)v;
    }
}

int main() {
    auto t0 = steady_clock::now();
    std::thread p(producer, 30, 10), c(consumer, 30, 50);
    p.join(); c.join();
    std::cout << "30個を処理するのにかかった時間 = "
              << duration_cast<milliseconds>(steady_clock::now() - t0).count() << " ms\n";
    std::cout << "キューに並んだ最大の個数 = " << q.peak() << " 個（容量は 3）\n";
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ex05c.cpp -o ex05c && ./ex05c

### 5-3-1. 結果 ―― 時間は変わらない。メモリだけが減る

- 所要時間 … 第2版とほぼ同じ（約1500ms）
- 並んだ最大の個数 … **24個 → 3個**

「入れる側を待たせたのだから、遅くなるはずだ」と予測した人が多いと思います。
**遅くなりません。** 理由は演習1で見たとおりです。

全体の速さを決めているのは**一番遅い段**（ここでは 50ms の受け取る側）です。
作る側がどれだけ先へ進もうと、受け取る側が 1個 50ms でしか処理できない以上、
30個に 1500ms かかることは変わりません。

先へ進んだ分は「速さ」にはならず、**キューに積み上がるだけ**でした。
容量を付けるとは、その**積み上がるはずだった分を作る側の待ち時間に変える**ことです。

```
【容量なし】
作る係     ■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■（先へ先へ）
キュー     ▁▂▃▄▅▆▇███████████▇▆▅▄▃▂▁          （24個まで伸びる）

【容量3】
作る係     ■■■...■...■...■...■...■...■...■   （満杯なら待たされる）
キュー     ▁▂▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃    （3個で頭打ち）
```

この「速い側が遅い側に合わせて待たされる」しくみを **バックプレッシャ（backpressure）**
と呼びます。**演習6**でくわしく扱います。

## 5-4. 完成版を読む

上のコードをもう一度、上から順に読んでください。この演習の目的はここです。

### 5-4-1. メンバの役割

- `q_` … 中身。`std::queue` そのもの
- `capacity_` … 上限。コンストラクタで決める
- `mtx_` … 鍵。`q_` を触る全員がこれを通る
- `can_pop_` … 「取り出せるようになった」を知らせる係
- `can_push_` … 「入れられるようになった」を知らせる係

**鍵は1本、条件変数は2本**です。守るデータは `q_` 1つなので鍵は1本で足ります。
待つ理由が「空」と「満杯」の2種類あるので、条件変数は2本要ります。

### 5-4-2. `push` と `pop` は鏡写し

```cpp
void push(...)                          T pop()
  wait(満杯でなくなるまで)                 wait(空でなくなるまで)
  入れる                                  取り出す
  can_pop_ を起こす                       can_push_ を起こす
```

**自分が待った条件の「反対側」を起こす**、と覚えてください。
`push` は「取り出したい人」を、`pop` は「入れたい人」を起こします。

### 5-4-3. `lk.unlock();` を挟んでいる理由

```cpp
q_.push(v);
lk.unlock();                // 先に鍵を開けてから
can_pop_.notify_one();      // 起こす
```

演習3の発展課題4で見た **hurry up and wait** を避けるためです。
鍵を握ったまま起こすと、起こされた人が鍵を取れずにもう一度眠ります。
`unique_lock` を使っているから、こう書けます。

### 5-4-4. `mutable std::mutex mtx_;` の `mutable`

`size()` は中身を変えないので `const` を付けたいのですが、
**鍵をかける操作は `mtx_` を変更します**。`const` メンバ関数の中では
メンバを変更できないので、そのままではコンパイルが通りません。

`mutable` は「このメンバだけは `const` の中でも変更してよい」という指定です。
`mutex` に `mutable` を付けるのは定型句なので、見たら「`const` な `size()` があるんだな」
と思ってください。

### 5-4-5. 外から見えるもの

`public:` にあるのは `push` / `pop` / `size` だけです。
`mtx_` も `can_pop_` も外からは触れません。

つまり **使う側は、鍵のことも条件変数のことも知らなくてよい**。
これがクラスに閉じ込めた最大の効果です。

> **並行処理の難しさは、閉じ込められる。**
> 難しいコードを「みんなが書く場所」から「1か所」に移すのが定石です。

## 発展課題

1. `pop()` には2つの形がありました。
   - 第1版 … `bool pop(T& out)` （空なら `false` を返す。待たない）
   - 完成版 … `T pop()` （空なら待つ。失敗しない）

   **どちらの形が必要になる場面**が、それぞれあります。具体例を1つずつ挙げてください。

2. 完成版の `push` から `lk.unlock();` の行を消したら、動作は変わるでしょうか。
   **正しさ**と**速さ**に分けて答えてください。

3. `size()` があるので、使う側でこう書けそうに見えます。

   ```cpp
   if (q.size() < 3) q.push(x);      // 3個未満のときだけ入れる
   ```

   `size()` も `push()` も、それぞれ鍵で正しく守られています。
   **それでもこのコードは期待どおりに動きません。** なぜでしょうか。

4. 条件変数を `can_pop_` と `can_push_` の2本ではなく、**1本にまとめた**らどうなるでしょうか。
   `notify_one` の場合と `notify_all` の場合で、それぞれ考えてください。

5. 容量を **1** にしたら、パイプライン全体はどうなるでしょうか。
   段と段が「同時に動ける」と言えるか、考えてみてください。

6. `push` の述語は `q_.size() < capacity_` です。
   もし `capacity_` を**あとから変更できる**ようにしたい場合、
   どこで変更し、変更したあとに何をしなければならないでしょうか。